In [1]:
import pymorphy3
import re
from collections import Counter

morph = pymorphy3.MorphAnalyzer()

In [2]:
with open('dom.txt', 'r', encoding='cp1251') as file:
    corpus = file.readlines()

In [3]:
def extract_clean_text(line):
    """Извлекает чистый текст из строки с тегами."""
    return ' '.join([word.split('>')[1].split('<')[0] for word in line.split() if '>' in word and '<' in word])

In [4]:
def format_results(results):
    """Форматирует результаты для вывода: одно предложение на строку."""
    return '\n'.join(results)

In [5]:
def count_token_frequency(token):
    """Подсчитывает частотность токена в корпусе."""
    counter = 0
    for line in corpus:
        if token in line:
            counter += 1
    return counter

In [6]:
def search_by_token(token, max_results=5):
    """Поиск строк, содержащих указанный токен."""
    results = []
    for line in corpus:
        if token in line:
            results.append(extract_clean_text(line))
            if len(results) >= max_results:
                break
    frequency = count_token_frequency(token)
    return f"Frequency: {frequency}\n{format_results(results)}"

In [7]:
def count_lemma_frequency(lemma):
    """Подсчитывает частотность леммы в корпусе."""
    counter = 0
    for line in corpus:
        words = line.split()
        for word in words:
            clean_word = word.split('>')[1].split('<')[0] if '>' in word and '<' in word else word
            parsed = morph.parse(clean_word.strip('.,!?\"\'\"'))[0]
            if parsed.normal_form == lemma:
                counter += 1
                break
    return counter

In [8]:
def search_by_lemma(lemma, max_results=5):
    """Поиск строк, содержащих указанный лемматизированный токен."""
    results = []
    for line in corpus:
        words = line.split()
        for word in words:
            clean_word = word.split('>')[1].split('<')[0] if '>' in word and '<' in word else word
            parsed = morph.parse(clean_word.strip('.,!?\"\'\"'))[0]
            if parsed.normal_form == lemma:
                results.append(extract_clean_text(line))
                break
        if len(results) >= max_results:
            break
    frequency = count_lemma_frequency(lemma)
    return f"Frequency: {frequency}\n{format_results(results)}"

In [9]:
def count_tag_frequency(tag):
    """Подсчитывает частотность семантического тега в корпусе."""
    counter = 0
    for line in corpus:
        if tag in line:
            counter += 1
    return counter

In [10]:
def search_by_semantic_tag(tag, max_results=5):
    """Поиск строк, содержащих указанный семантический тег."""
    results = []
    for line in corpus:
        words = line.split()
        for word in words:
            # Проверяем, содержит ли слово указанный тег внутри атрибута sem
            if re.search(f"sem=['\"][^'\"]*{tag}[^'\"]*['\"]", word):
                # Извлекаем само слово с тегом
                clean_word = word.split('>')[1].split('<')[0]
                # Форматируем результат: сначала слово, затем контекст
                context = extract_clean_text(line)
                results.append(f"Word: {clean_word}\nContext: {context}")
                break  # Выход из цикла, чтобы не искать в этом предложении больше
        if len(results) >= max_results:
            break  # Ограничиваем количество выводимых результатов
    frequency = count_tag_frequency(tag)
    return f"Frequency: {frequency}\n{format_results(results)}"

In [11]:
def count_tag_combination_frequency(tags):
    """Подсчитывает частотность комбинации тегов в корпусе."""
    counter = 0
    for line in corpus:
        if all(tag in line for tag in tags):
            counter += 1
    return counter

In [12]:
def search_by_tag_combination(tags, max_results=5):
    """Поиск строк, содержащих комбинацию тегов."""
    results = []
    for line in corpus:
        # Проверяем, содержатся ли все теги в строке с помощью регулярных выражений
        if all(re.search(f"sem=['\"][^'\"]*{tag}[^'\"]*['\"]", line) for tag in tags):
            words_with_tags = []
            words = line.split()
            for word in words:
                # Используем регулярные выражения для поиска нужных тегов внутри атрибута sem
                for tag in tags:
                    if re.search(f"sem=['\"][^'\"]*{tag}[^'\"]*['\"]", word):
                        clean_word = word.split('>')[1].split('<')[0]
                        words_with_tags.append(f"Word: {clean_word}")
            if words_with_tags:
                # Если найдены слова с нужными тегами, добавляем их в результат
                context = extract_clean_text(line)
                for word in words_with_tags:
                    results.append(f"{word}\nContext: {context}")
            if len(results) >= max_results:
                break
    frequency = count_tag_combination_frequency(tags)
    return f"Frequency: {frequency}\n{format_results(results)}"

In [13]:
if __name__ == "__main__":
    token_name = 'рижском'
    token_results = search_by_token(token_name, max_results=3)
    print(f"Contexts for token '{token_name}':\n{token_results}")

Contexts for token 'рижском':
Frequency: 1
  В  рижском  спектакле  сценография  Ф  Вогербауэра  была  лаконична  но  всё  же  воссоздавала  гумилевскую  атмосферу  старой  заводи  где  есть дом с  голубыми  ставнями  с  креслами  давними  а  мелодии  латышского  маэстро  Р  Паулса  и  итальянского  А  Аннеккино  были  вполне  созвучны  настроению  как  писали  в  старину  Треплева  шопеновских  вальсов 


In [14]:
if __name__ == "__main__":
    lemma_name = 'рижский'
    lemma_results = search_by_lemma(lemma_name, max_results=3)
    print(f"Contexts for lemma '{lemma_name}':\n{lemma_results}")

Contexts for lemma 'рижский':
Frequency: 2
  В  рижском  спектакле  сценография  Ф  Вогербауэра  была  лаконична  но  всё  же  воссоздавала  гумилевскую  атмосферу  старой  заводи  где  есть дом с  голубыми  ставнями  с  креслами  давними  а  мелодии  латышского  маэстро  Р  Паулса  и  итальянского  А  Аннеккино  были  вполне  созвучны  настроению  как  писали  в  старину  Треплева  шопеновских  вальсов 
  В  июле  Кирилл   уехал  со  студенческим  отрядом  в  Новгород  а  мы  с  Ритой  в  конце  июля  взяли  путёвки  на  Рижское  взморье  поехали  немного  раньше  пожили  в  гостинице  а  с  августа      поселились  в доме отдыха 


In [15]:
if __name__ == "__main__":
    semantic_tag_name = 'r:qual'
    semantic_results = search_by_semantic_tag(semantic_tag_name, max_results=3)
    print(f"Contexts for semantic tag '{semantic_tag_name}':\n{semantic_results}")

Contexts for semantic tag 'r:qual':
Frequency: 1396
Word: побогаче
Context:   Должники  возвращали  долг  тем  что  приводили  в дом к  Бертеньеву  новых  клиентов  из  тех  что  были  уже  покрупнее  побогаче
Word: сложным
Context:   Прикрепив  на  разной  высоте  полусферические  контейнеры  с  ампельными  растениями  непосредственно  к  стене дома или  ограде  можно  не  прибегая  к  сложным  конструкциям  декорировать  достаточно  большие  плоскости 
Word: собственный
Context:   Ваши  родственники  или  знакомые  затеяли  строить  собственный дом


In [16]:
if __name__ == "__main__":
    tag_combination_name = ["r:rel", "t:constr"]
    combination_results = search_by_tag_combination(tag_combination_name, max_results=3)
    print(f"Contexts for tag combination '{', '.join(tag_combination_name)}':\n{combination_results}")

Contexts for tag combination 'r:rel, t:constr':
Frequency: 1861
Word: лиственных
Context:   Например  декорирование  фасада дома  беседки  или  даже  ограды  шпалерами  из  хвойных  и  лиственных  растений 
Word: древнем
Context:   Во  время  работы  жюри  работы  были  выставлены  в Доме Европы  затем  перекочевали  на  недельку  во  Дворец  творчества  детей  и  юношества  на  Воробьевых  горах  а  сегодня  выставка  этих  работ  наконец  то  открывается  в  помещении  самого  музея  в  Щетининском  переулке  в  древнем  Замоскворечье 
Word: декоративными
Context:   От  средневекового  города  осталось  лишь  несколько  кварталов  с  декоративными  бревенчатыми домами  собор  Святого  Петра  Дворец  правосудия 
